# Test Pipeline RAG — Assessment Jawaban Interview

Notebook ini menguji pipeline RAG secara bertahap, dari yang gratis ke yang memanggil LLM:

| Tahap | Yang diuji | Biaya |
|---|---|---|
| 0 | Environment & LLM provider | gratis |
| 1 | Koneksi Qdrant & isi collection | gratis |
| 2 | Embedding (dimensi & pemotongan token) | gratis |
| 3 | Kualitas retrieval (hit@1, hit@3, MRR) | gratis |
| 4 | Konteks referensi yang dikirim ke LLM | gratis |
| 5 | End-to-end assessment (smoke test) | 1 assessment |
| 6 | Konsistensi & daya beda skor | `3 × N_RUNS` assessment |

**Cara menjalankan:** pilih kernel dari `.venv` project, lalu *Run All*. Kalau satu tahap gagal,
perbaiki dulu sebelum lanjut — tahap berikutnya bergantung pada tahap sebelumnya.

LLM dipilih lewat `.env`:
- `LLM_PROVIDER=ollama` (default) — model lokal, atur dengan `OLLAMA_MODEL` (default `qwen3:4b`).
  Ollama harus sudah jalan dan modelnya sudah di-`pull`.
- `LLM_PROVIDER=openrouter` — atur dengan `OPENROUTER_MODEL` (default model nemotron `:free`).

Dengan Ollama, tahap 5–6 gratis dan tidak terkena batas kuota.

## Konfigurasi

In [1]:
RUN_LLM = True   # False = hanya jalankan tahap 0-4 (tanpa memanggil LLM)
N_RUNS = 3       # berapa kali tiap jawaban dinilai di tahap 6
TOP_K = 3        # jumlah referensi yang diambil retrieval (sama dengan default RAG.py)

## Tahap 0 — Environment

In [2]:
import os
import sys
import json
import statistics
from pathlib import Path

# RAG.py memakai import datar (`from qdrant import ...`), jadi folder rag/
# harus jadi working directory dan ada di sys.path.
if Path.cwd().name != "rag" and (Path.cwd() / "rag").is_dir():
    os.chdir(Path.cwd() / "rag")
RAG_DIR = Path.cwd()
if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))

from dotenv import load_dotenv, find_dotenv
env_file = find_dotenv(usecwd=True)
load_dotenv(env_file)
print("Working dir:", RAG_DIR)
print(".env       :", env_file or "TIDAK DITEMUKAN")

def mask(v):
    return "(kosong)" if not v else f"{v[:4]}...{v[-4:]} ({len(v)} karakter)"

provider = os.getenv("LLM_PROVIDER", "ollama").lower()
env_status = {
    "cluster_endpoint": os.getenv("cluster_endpoint"),
    "api_qdrant_interview": os.getenv("api_qdrant_interview"),
}
if provider == "openrouter":
    env_status["OPENROUTER_API_KEY / openrouter_api_key"] = (
        os.getenv("OPENROUTER_API_KEY") or os.getenv("openrouter_api_key")
    )
for k, v in env_status.items():
    print(f"  {k:42s} {mask(v)}")
print(f"  {'LLM_PROVIDER':42s} {provider}")

missing = [k for k, v in env_status.items() if not v]
assert not missing, f"Env belum diisi: {missing}"

if provider == "ollama":
    import requests
    base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    model = os.getenv("OLLAMA_MODEL", "qwen3:4b")
    try:
        tags = requests.get(f"{base_url}/api/tags", timeout=5).json()
    except requests.ConnectionError:
        raise AssertionError(f"Ollama tidak bisa dihubungi di {base_url}. Sudah dijalankan?") from None
    local_models = [m["name"] for m in tags.get("models", [])]
    print(f"  {'Ollama':42s} {base_url}")
    print(f"  {'Model terpasang':42s} {local_models or '-'}")
    assert model in local_models or f"{model}:latest" in local_models, (
        f"Model '{model}' belum ada. Jalankan: ollama pull {model}"
    )

Working dir: d:\sistem-evaluasi-pemahaman-mahasiswa\rag
.env       : d:\sistem-evaluasi-pemahaman-mahasiswa\.env
  cluster_endpoint                           http...t.io (79 karakter)
  api_qdrant_interview                       eyJh...jokU (176 karakter)
  LLM_PROVIDER                               ollama
  Ollama                                     http://localhost:11434
  Model terpasang                            ['qwen3.5:4b']


## Tahap 1 — Koneksi Qdrant & isi collection

Dicek dengan client terpisah **sebelum** meng-import `qdrant.py`, karena `qdrant.py` otomatis membuat
collection *kosong* kalau belum ada — retrieval jadi diam-diam mengembalikan nol hasil.

In [3]:
from qdrant_client import QdrantClient

COLLECTION_NAME = "interview_knowledge"
CHUNKS_FILE = RAG_DIR / "chunking_etika" / "all_etika_chunks.jsonl"

raw_client = QdrantClient(url=os.getenv("cluster_endpoint"), api_key=os.getenv("api_qdrant_interview"))
assert raw_client.collection_exists(COLLECTION_NAME), (
    f"Collection '{COLLECTION_NAME}' tidak ada di Qdrant. Jalankan notebook ingesting dulu."
)

info = raw_client.get_collection(COLLECTION_NAME)
n_points = raw_client.count(COLLECTION_NAME, exact=True).count
local_chunks = [json.loads(l) for l in CHUNKS_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]
chunk_by_id = {c["chunk_id"]: c for c in local_chunks}

print("Vector size :", info.config.params.vectors.size)
print("Distance    :", info.config.params.vectors.distance)
print("Jumlah point:", n_points, "| jumlah chunk lokal:", len(local_chunks))

points, _ = raw_client.scroll(COLLECTION_NAME, limit=10_000, with_payload=True, with_vectors=False)
remote_ids = {p.payload.get("chunk_id") for p in points}
only_local = sorted(set(chunk_by_id) - remote_ids)
only_remote = sorted(remote_ids - set(chunk_by_id))
print("Chunk hanya di lokal :", only_local or "-")
print("Chunk hanya di Qdrant:", only_remote or "-")

assert info.config.params.vectors.size == 384, "Dimensi vector tidak cocok dengan model embedding (384)."
assert n_points > 0, "Collection kosong."
if n_points != len(local_chunks) or only_local or only_remote:
    print("\n[PERINGATAN] Isi Qdrant tidak sama dengan file chunk lokal — pertimbangkan ingest ulang.")
else:
    print("\nOK: isi Qdrant sama dengan file chunk lokal.")

d:\sistem-evaluasi-pemahaman-mahasiswa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector size : 384
Distance    : Cosine
Jumlah point: 53 | jumlah chunk lokal: 53
Chunk hanya di lokal : -
Chunk hanya di Qdrant: -

OK: isi Qdrant sama dengan file chunk lokal.


## Tahap 2 — Embedding

Model `paraphrase-multilingual-MiniLM-L12-v2` memotong input di **128 token**. Teks setelah token ke-128
tidak ikut ter-embed, sehingga isi chunk yang panjang tidak bisa ditemukan lewat pencarian.

In [4]:
from tokenizers import Tokenizer
from embed import embedding_model, create_embedding

vec = create_embedding("Mengapa transparansi penting dalam kecerdasan artifisial?")
print("Dimensi embedding:", len(vec))
assert len(vec) == 384

tok = embedding_model.model.tokenizer
max_len = tok.truncation["max_length"] if tok.truncation else None
full_tok = Tokenizer.from_str(tok.to_str())
full_tok.no_truncation()

rows = []
for c in local_chunks:
    n = len(full_tok.encode(c["content"]).ids)
    rows.append((c["chunk_id"], c["title"][:45], n, min(1.0, max_len / n)))

truncated = [r for r in rows if r[2] > max_len]
print(f"Batas token: {max_len}")
print(f"Chunk yang terpotong: {len(truncated)} dari {len(rows)}\n")
print(f"{'chunk_id':22s} {'title':45s} {'token':>6s} {'ter-embed':>9s}")
for cid, title, n, cov in sorted(truncated, key=lambda r: r[3]):
    print(f"{cid:22s} {title:45s} {n:6d} {cov:8.0%}")

Dimensi embedding: 384
Batas token: 128
Chunk yang terpotong: 35 dari 53

chunk_id               title                                          token ter-embed
modul_etika_ai_33      Kualitas Data dalam KA                          1287      10%
modul_etika_ai_35      Contoh Aktivitas Penerapan KA Generatif dalam   1258      10%
modul_etika_ai_44      Teknologi Deepfake                               981      13%
modul_etika_ai_34      Implementasi KA Generatif pada Kehidupan Seha    896      14%
pedoman_etika_35       Matriks Usulan Upaya Mitigasi Risiko di Setia    860      15%
pedoman_etika_12       Konsep Manusia sebagai Sentra Kepentingan, Ke    799      16%
pedoman_etika_15       Transparansi, Akuntabilitas, dan Kejelasan Ri    786      16%
modul_etika_ai_45      Contoh Aktivitas Menganalisis konten Deepfake    753      17%
modul_etika_ai_31      Ruang Lingkup dan Definisi Teknologi KA Gener    720      18%
modul_etika_ai_32      Prinsip Kerja Teknologi KA Generatif             683

## Tahap 3 — Kualitas retrieval

Golden set: pertanyaan + chunk yang **seharusnya** ditemukan (boleh lebih dari satu chunk yang benar).
Tambah / ubah daftar `GOLDEN` sesuai pertanyaan interview yang sebenarnya dipakai.

- **hit@1** — chunk benar ada di peringkat 1
- **hit@k** — chunk benar ada di top-`TOP_K` (yang benar-benar dikirim ke LLM)
- **MRR** — rata-rata 1/peringkat chunk benar pertama (dicari sampai top-10)

In [5]:
from qdrant import retrieve_knowledge

GOLDEN = [
    ("Mengapa transparansi penting dalam penerapan kecerdasan artifisial?", {"modul_etika_ai_38", "pedoman_etika_15"}),
    ("Apa yang dimaksud dengan kecerdasan artifisial generatif?",           {"modul_etika_ai_31", "modul_etika_ai_30"}),
    ("Bagaimana prinsip kerja teknologi KA generatif?",                       {"modul_etika_ai_32"}),
    ("Mengapa kualitas data penting dalam kecerdasan artifisial?",            {"modul_etika_ai_33", "pedoman_etika_32"}),
    ("Berikan contoh penerapan KA generatif dalam kehidupan sehari-hari.",    {"modul_etika_ai_34"}),
    ("Bagaimana KA generatif dapat dimanfaatkan dalam bidang pendidikan?",    {"modul_etika_ai_35", "pedoman_etika_27"}),
    ("Siapa pemilik hak cipta atas karya yang dihasilkan AI?",                {"modul_etika_ai_39"}),
    ("Bagaimana KA generatif dapat disalahgunakan untuk menyebarkan hoaks?",  {"modul_etika_ai_41"}),
    ("Bagaimana bias dapat muncul pada data dan hasil KA?",                   {"modul_etika_ai_42", "pedoman_etika_18"}),
    ("Apa risiko KA terhadap privasi dan keamanan data pribadi?",             {"modul_etika_ai_43", "pedoman_etika_14"}),
    ("Apa itu deepfake dan bagaimana cara mengenalinya?",                     {"modul_etika_ai_44", "modul_etika_ai_45"}),
    ("Bagaimana nilai-nilai Pancasila menjadi dasar etika KA?",               {"pedoman_etika_06", "pedoman_etika_10"}),
    ("Mengapa manusia harus tetap memegang kendali dan pengawasan atas KA?",  {"pedoman_etika_12"}),
    ("Apa saja aspek keamanan teknis yang perlu diperhatikan dalam sistem KA?", {"pedoman_etika_13"}),
    ("Siapa yang bertanggung jawab secara hukum jika sistem KA merugikan pengguna?", {"pedoman_etika_23", "pedoman_etika_19"}),
    ("Apa peran universitas dalam pengembangan KA yang beretika?",            {"pedoman_etika_27"}),
    ("Apa peran pemerintah dalam tata kelola kecerdasan artifisial?",         {"pedoman_etika_26"}),
    ("Mengapa sistem KA harus dapat diaudit dan ditelusuri?",                 {"pedoman_etika_24"}),
    ("Bagaimana upaya mitigasi risiko di setiap tahapan pengembangan KA?",    {"pedoman_etika_35", "pedoman_etika_15"}),
]

unknown = {cid for _, exp in GOLDEN for cid in exp} - set(chunk_by_id)
assert not unknown, f"chunk_id di GOLDEN tidak ada di file chunk: {unknown}"

retrieval_results = []
for q, expected in GOLDEN:
    hits = retrieve_knowledge(q, limit=10)
    got = [h.payload.get("chunk_id") for h in hits]
    rank = next((i for i, cid in enumerate(got, 1) if cid in expected), None)
    retrieval_results.append({"question": q, "expected": expected, "got": got,
                              "scores": [h.score for h in hits], "rank": rank})

n = len(retrieval_results)
hit1 = sum(r["rank"] == 1 for r in retrieval_results) / n
hitk = sum(r["rank"] is not None and r["rank"] <= TOP_K for r in retrieval_results) / n
mrr = sum(1 / r["rank"] for r in retrieval_results if r["rank"]) / n

for r in retrieval_results:
    status = "OK  " if r["rank"] and r["rank"] <= TOP_K else "MISS"
    print(f"[{status}] rank={str(r['rank'] or '-'):>2}  {r['question']}")
print(f"\nhit@1 = {hit1:.0%}   hit@{TOP_K} = {hitk:.0%}   MRR@10 = {mrr:.2f}   (n={n})")

Collection sudah tersedia.
[OK  ] rank= 1  Mengapa transparansi penting dalam penerapan kecerdasan artifisial?
[OK  ] rank= 1  Apa yang dimaksud dengan kecerdasan artifisial generatif?
[OK  ] rank= 1  Bagaimana prinsip kerja teknologi KA generatif?
[OK  ] rank= 1  Mengapa kualitas data penting dalam kecerdasan artifisial?
[OK  ] rank= 1  Berikan contoh penerapan KA generatif dalam kehidupan sehari-hari.
[MISS] rank= 9  Bagaimana KA generatif dapat dimanfaatkan dalam bidang pendidikan?
[OK  ] rank= 1  Siapa pemilik hak cipta atas karya yang dihasilkan AI?
[OK  ] rank= 2  Bagaimana KA generatif dapat disalahgunakan untuk menyebarkan hoaks?
[OK  ] rank= 1  Bagaimana bias dapat muncul pada data dan hasil KA?
[OK  ] rank= 1  Apa risiko KA terhadap privasi dan keamanan data pribadi?
[OK  ] rank= 1  Apa itu deepfake dan bagaimana cara mengenalinya?
[OK  ] rank= 3  Bagaimana nilai-nilai Pancasila menjadi dasar etika KA?
[OK  ] rank= 2  Mengapa manusia harus tetap memegang kendali dan pengawasa

In [6]:
# Detail pertanyaan yang gagal: chunk apa yang justru terambil?
misses = [r for r in retrieval_results if not (r["rank"] and r["rank"] <= TOP_K)]
if not misses:
    print("Semua pertanyaan menemukan chunk yang benar di top-%d." % TOP_K)
for r in misses:
    print("=" * 90)
    print("Q       :", r["question"])
    print("Harusnya:", ", ".join(f"{cid} ({chunk_by_id[cid]['title']})" for cid in sorted(r["expected"])))
    for cid, score in list(zip(r["got"], r["scores"]))[:TOP_K]:
        print(f"  dapat : {score:.3f}  {cid:20s} {chunk_by_id.get(cid, {}).get('title', '?')}")

Q       : Bagaimana KA generatif dapat dimanfaatkan dalam bidang pendidikan?
Harusnya: modul_etika_ai_35 (Contoh Aktivitas Penerapan KA Generatif dalam Bidang Pendidikan), pedoman_etika_27 (Peran Universitas/Institusi Pendidikan)
  dapat : 0.731  modul_etika_ai_25    Capaian Pelatihan
  dapat : 0.698  modul_etika_ai_37    Etika dalam Penggunaan KA Generatif
  dapat : 0.691  modul_etika_ai_34    Implementasi KA Generatif pada Kehidupan Sehari-hari
Q       : Siapa yang bertanggung jawab secara hukum jika sistem KA merugikan pengguna?
Harusnya: pedoman_etika_19 (Kewaspadaan/Kehati-hatian/Awareness AI/Pertanggungjawaban), pedoman_etika_23 (Kejelasan Pertanggungjawaban Hukum Kepada Pengguna)
  dapat : 0.664  pedoman_etika_24     Auditable (Dapat Ditelusuri dengan Baik)
  dapat : 0.641  modul_etika_ai_43    Pelanggaran Privasi dan Keamanan Data
  dapat : 0.627  modul_etika_ai_38    Transparansi dan Keadilan
Q       : Apa peran universitas dalam pengembangan KA yang beretika?
Harusnya: pedoma

## Tahap 4 — Konteks referensi (teks yang benar-benar dikirim ke LLM)

In [7]:
import RAG  # memuat model saja, belum ada panggilan LLM

print(f"LLM: {RAG.LLM_PROVIDER} / {RAG.MODEL_NAME} (temperature={RAG.TEMPERATURE})")
context = RAG.format_references(RAG.retrieve(GOLDEN[0][0], top_k=TOP_K))
print(f"Panjang konteks: {len(context)} karakter\n")
print(context[:3000] + ("\n... (dipotong)" if len(context) > 3000 else ""))

LLM: ollama / qwen3.5:4b (temperature=0.0)
Panjang konteks: 2178 karakter


Reference 1

Section:
C.1.1

Title:
Transparansi dan Keadilan

Content:
C.1.1. Transparansi dan Keadilan 
Transparansi dalam penggunaan KA berarti bahwa pengguna harus mengetahui cara 
kerja sistem, bagaimana data digunakan, dan bagaimana hasilnya dihasilkan. Ini penting 
agar pengguna tidak tertipu oleh informasi yang dibuat oleh KA dan dapat memahami 
batasan teknologi ini. 
Keadilan dalam KA berarti bahwa teknologi ini harus digunakan secara setara, tanpa 
memihak atau merugikan kelompok tertentu.

Similarity Score:
0.7214857

--------------------------------

Reference 2

Section:
C.a

Title:
Andal

Content:
a. Andal
Sistem Kecerdasan Artifisial bekerja dengan akurat terhadap situasi/input yang nyata (real world 
situation/inputs), terukur dan sesuai dengan tujuan perancangannya (predictability), keakuratan 
tidak hanya terhadap input data pelatihan saja. Sistem Kecerdasan Artifisial yang (dihasilkan 
dan) 

## Tahap 5 — End-to-end smoke test

Yang dicek: referensi ikut terambil, dan hasilnya berupa `AssessmentResult` yang valid
(skor 0–100 sudah divalidasi Pydantic) dengan semua field terisi.

In [8]:
import time
from schemas import AssessmentResult

def assess(question, answer):
    t0 = time.perf_counter()
    response = RAG.run_assessment(question, answer, top_k=TOP_K)
    result = response["result"]
    return {"result": result, "score": result.score, "references": response["references"],
            "seconds": time.perf_counter() - t0}

if RUN_LLM:
    q = "Mengapa transparansi penting dalam penerapan kecerdasan artifisial?"
    a = "Karena agar tidak terjadi kesalahpahaman atau bias terhadap suatu permasalahan atau sebuah kesimpulan."
    smoke = assess(q, a)
    result = smoke["result"]
    print(result.model_dump_json(indent=2))
    print("\n" + "-" * 60)
    print("Referensi :", [r["chunk_id"] for r in smoke["references"]])
    print(f"Durasi    : {smoke['seconds']:.1f} detik")
    assert smoke["references"], "Tidak ada referensi terambil — penilaian tanpa referensi."
    assert isinstance(result, AssessmentResult), "Hasil bukan AssessmentResult."
    empty = [f for f in ("feedback", "strengths", "weaknesses", "recommendation") if not getattr(result, f)]
    assert not empty, f"Field kosong: {empty}"
    print("OK: referensi terambil, semua field terisi.")
else:
    print("RUN_LLM = False, tahap dilewati.")

{
  "feedback": "Jawaban mahasiswa menyentuh inti masalah (bias dan kesalahpahaman) yang merupakan aspek krusial dari transparansi, namun penjelasannya masih sangat singkat dan kurang mendalam. Jawaban ini belum sepenuhnya menjawab 'mengapa' secara komprehensif berdasarkan referensi. \n\nPoin utama yang terjawab:\n1. Mencegah bias terhadap kesimpulan/permasalahan.\n2. Mencegah kesalahpahaman.\nPoin utama yang kurang/tidak terjawab dengan baik:\n1. Hak pengguna untuk mengetahui cara kerja sistem dan penggunaan data (dari Ref 1).\n2. Kemampuan untuk memahami batasan teknologi agar tidak tertipu (dari Ref 1).\n3. Konsep keadilan/keberpihakan terhadap kelompok tertentu (dari Ref 1 & Jawaban mahasiswa, tapi penjelasannya kurang elaborasi).",
  "strengths": [
    "Mengidentifikasi isu bias sebagai dampak negatif dari kurangnya transparansi.",
    "Menyebutkan risiko kesalahpahaman akibat ketidaktransparanan.",
    "Bahasa yang digunakan singkat dan langsung pada inti masalah."
  ],
  "weakne

## Tahap 6 — Konsistensi & daya beda skor

Satu pertanyaan dengan tiga kualitas jawaban, masing-masing dinilai `N_RUNS` kali.
Harapannya: rata-rata skor **bagus > sedang > buruk**, dan rentang skor per jawaban kecil.
Rentang besar menandakan penilaian kurang stabil (misalnya karena `temperature` tinggi).

Kalau memakai OpenRouter, model `:free` punya batas request per menit/hari; run yang gagal dicatat,
tidak menghentikan loop.

In [9]:
QUESTION = "Mengapa transparansi penting dalam penerapan kecerdasan artifisial?"
ANSWERS = {
    "bagus": (
        "Transparansi penting agar pengguna dan pihak terdampak dapat memahami bagaimana sistem KA "
        "mengambil keputusan, data apa yang dipakai, serta apa keterbatasan dan risikonya. Dengan "
        "transparansi, keputusan KA bisa dijelaskan, diaudit, dan dipertanggungjawabkan, sehingga bias "
        "atau kesalahan dapat dideteksi dan diperbaiki. Transparansi juga membangun kepercayaan publik "
        "dan mendukung keadilan, misalnya dengan memberi tahu pengguna bahwa suatu konten dibuat oleh KA."
    ),
    "sedang": (
        "Supaya orang tahu cara kerja AI dan bisa percaya pada hasilnya, serta kalau ada kesalahan "
        "bisa diketahui."
    ),
    "buruk": "AI itu canggih dan bisa membantu pekerjaan manusia jadi lebih cepat.",
}

scores = {k: [] for k in ANSWERS}
errors = []
if RUN_LLM:
    for label, answer in ANSWERS.items():
        for i in range(N_RUNS):
            try:
                r = assess(QUESTION, answer)
                scores[label].append(r["score"])
                print(f"{label:6s} run {i + 1}: score={r['score']}  ({r['seconds']:.1f} detik)")
            except Exception as e:
                errors.append((label, i + 1, f"{type(e).__name__}: {e}"))
                print(f"{label:6s} run {i + 1}: ERROR {type(e).__name__}: {str(e)[:150]}")
else:
    print("RUN_LLM = False, tahap dilewati.")

bagus  run 1: score=95  (15.3 detik)
bagus  run 2: score=95  (14.5 detik)
bagus  run 3: score=95  (15.4 detik)
sedang run 1: score=65  (12.8 detik)
sedang run 2: score=65  (12.8 detik)
sedang run 3: score=65  (13.9 detik)
buruk  run 1: score=15  (14.3 detik)
buruk  run 2: score=15  (13.9 detik)
buruk  run 3: score=15  (16.2 detik)


In [10]:
if RUN_LLM:
    print(f"{'jawaban':8s} {'skor':24s} {'rata2':>6s} {'rentang':>8s}")
    means = {}
    for label, vals in scores.items():
        valid = [v for v in vals if v is not None]
        if valid:
            means[label] = statistics.mean(valid)
            print(f"{label:8s} {str(vals):24s} {means[label]:6.1f} {max(valid) - min(valid):8d}")
        else:
            print(f"{label:8s} {str(vals):24s} {'-':>6s} {'-':>8s}")

    if len(means) == 3:
        ordered = means["bagus"] > means["sedang"] > means["buruk"]
        print("\nUrutan bagus > sedang > buruk:", "OK" if ordered else "TIDAK SESUAI")
    if errors:
        print(f"\n{len(errors)} run gagal (cek rate limit / kredit OpenRouter).")

jawaban  skor                      rata2  rentang
bagus    [95, 95, 95]               95.0        0
sedang   [65, 65, 65]               65.0        0
buruk    [15, 15, 15]               15.0        0

Urutan bagus > sedang > buruk: OK
